# Experimento CIFAR-10 con CNN

Este notebook muestra la carga, visualización y partición del dataset CIFAR-10.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

SEED = 76
torch.manual_seed(SEED)
np.random.seed(SEED)

root = './data'
transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
raw_train_set = datasets.CIFAR10(root=root, train=True, download=True)
test_set = datasets.CIFAR10(root=root, train=False, download=True, transform=eval_transform)

labels = np.array(raw_train_set.targets)
train_idx, val_idx = train_test_split(np.arange(len(raw_train_set)), test_size=0.2, stratify=labels, random_state=SEED)
class TransformedDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, image_transform):
        self.dataset = dataset
        self.image_transform = image_transform
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, index):
        image, label = self.dataset[index]
        return self.image_transform(image), label

train_set = torch.utils.data.Subset(TransformedDataset(raw_train_set, transform), train_idx)
val_set = torch.utils.data.Subset(TransformedDataset(raw_train_set, eval_transform), val_idx)
print('Train size:', len(train_idx))
print('Validation size:', len(val_idx))
print('Test size:', len(test_set))

class_names = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
fig, axes = plt.subplots(2, 4, figsize=(12, 7))
for ax in axes.flat:
    ax.axis('off')
for i in range(8):
    image, label = train_set[i]
    image = image.permute(1, 2, 0).numpy()
    mean = np.array([0.4914, 0.4822, 0.4465])
    std = np.array([0.2023, 0.1994, 0.2010])
    image = image * std + mean
    image = np.clip(image, 0, 1)
    row, col = divmod(i, 4)
    axes[row, col].imshow(image)
    axes[row, col].set_title(class_names[label])
plt.tight_layout()
plt.show()
